# Mockture: Server Object Usage

Mockture is a contract-aware in-process HTTP mock server. Its core `Mockture` class starts a local
HTTP server, registers response templates, and validates every interaction against an OpenAPI contract
— all in pure Python, without any pytest integration required.

When testing an HTTP client you need a controlled server that returns predictable responses. Mockture
goes further than a plain stub: it raises `ContractConfigError` immediately if you configure a
response body that violates the schema, and in strict mode it blocks malformed incoming requests with
HTTP 500. This catches contract mismatches at development time rather than in production.

After completing this notebook you will be able to configure a `Mockture` instance with templates,
override template arguments at call time, share arguments across multiple responses with
`for_context()`, and verify contract compliance using `assert_no_contract_violations()`. Each section
includes a **Plugin equivalent** block showing the identical operation via `@pytest.mark.mockture`
for when the pytest plugin is available. See [`example/`](../example/) for full plugin demos.

## Table of Contents

- [Prerequisites](#Prerequisites)
- [1. Setup](#1-Setup)
  - [1.1 Install Dependencies](#11-Install-Dependencies)
  - [1.2 Import Packages and Configure the Demo API](#12-Import-Packages-and-Configure-the-Demo-API)
  - [1.3 Start the Demo API](#13-Start-the-Demo-API)
  - [1.4 Verify API Behavior](#14-Verify-API-Behavior)
  - [1.5 Export the OpenAPI Contract](#15-Export-the-OpenAPI-Contract)
- [2. Templates](#2-Templates)
  - [2.1 Inspect the Templates File](#21-Inspect-the-Templates-File)
- [3. Template Defaults](#3-Template-Defaults)
- [4. Arg Overrides and Method Chaining](#4-Arg-Overrides-and-Method-Chaining)
- [5. Scoped Context](#5-Scoped-Context)
- [6. Inline Scenario Dict](#6-Inline-Scenario-Dict)
- [7. Contract Validation](#7-Contract-Validation)
  - [7.1 Config-Time Validation](#71-Config-Time-Validation)
  - [7.2 Runtime Strict Mode](#72-Runtime-Strict-Mode)
- [8. Non-Strict Mode](#8-Non-Strict-Mode)
- [9. Inspecting Recorded Calls](#9-Inspecting-Recorded-Calls)
- [10. Teardown](#10-Teardown)
- [11. Conclusion](#11-Conclusion)

## Prerequisites

No environment variables are required.

Additional prerequisites:

- `mockture` installed in the active environment (run `%pip install -e .[usage]` in the first cell).
- `uvicorn` and `httpx` available (included in the `[usage]` extra).
- `example/basic_api.py` present in the repository root — the notebook imports `app` from it.
- `orders.templates.yml` present in the `usage/` working directory.
- `utils.py` must be present in the parent directory and must export `display_citation` for
  rendering retrieved or generated text as formatted blockquotes.

## 1. Setup

### 1.1 Install Dependencies

We install the `mockture` package and its `[usage]` extras into the current kernel.

In [ ]:
%pip install -e .[usage]

Note: you may need to restart the kernel to use updated packages.


### 1.2 Import Packages and Configure the Demo API

We import the standard library, third-party, and Mockture modules, then load the demo FastAPI
application that serves as the real API reference for this notebook.

In [ ]:
import asyncio
import sys
from pathlib import Path

import httpx
import uvicorn
import yaml
from IPython.display import Markdown, display

repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

from mockture import Mockture
from mockture.errors import ContractConfigError
from example.basic_api import app

### 1.3 Start the Demo API

We launch the demo FastAPI application in the background using `uvicorn` and wait until it reports
that it has started before proceeding.

In [ ]:
host = "127.0.0.1"
port = 8010

config = uvicorn.Config(app=app, host=host, port=port, log_level='error')
server = uvicorn.Server(config)
server_task = asyncio.create_task(server.serve())

for _ in range(50):
    if server.started:
        break
    await asyncio.sleep(0.1)

if not server.started:
    raise RuntimeError("Demo API did not start in time")

print("Real API running at", f"http://{host}:{port}")

### 1.4 Verify API Behavior

We send two requests to the real API to confirm it is running and to observe its behavior: one
with a valid payload and one with an invalid payload.

In [ ]:
async with httpx.AsyncClient(base_url=f"http://{host}:{port}", timeout=5.0) as client:
    valid_order   = await client.post("/orders", json={"item_id": "SKU-42", "quantity": 2})
    invalid_order = await client.post("/orders", json={"item_id": "",       "quantity": 0})

print("POST /orders valid   ->", valid_order.status_code,   valid_order.json())
print("POST /orders invalid ->", invalid_order.status_code, invalid_order.json())

### 1.5 Export the OpenAPI Contract

We retrieve the OpenAPI schema from the running API and save it to `basic_api.swagger.yml`.
This file is used as the `contract_path` for every `Mockture` instance in subsequent sections.

In [ ]:
async with httpx.AsyncClient(base_url=f"http://{host}:{port}", timeout=5.0) as client:
    openapi_json = (await client.get("/openapi.json")).json()

swagger_path = Path("basic_api.swagger.yml")
swagger_path.write_text(yaml.safe_dump(openapi_json, sort_keys=False), encoding='utf-8')
print(f"Exported -> {swagger_path}")

## 2. Templates

### 2.1 Inspect the Templates File

Templates are defined in `orders.templates.yml`. We load `swagger_path` and `templates_path` as
`Path` objects and display the full templates file so the structure is visible before use.

Each template entry has two keys:

- **`args`**: default values for every placeholder used in `interaction`. A `null` value marks
  the argument as required — it must be supplied at `respond()` time.
- **`interaction`**: the HTTP method, path, and response shape, with `{placeholder}` tokens
  that are filled by resolving `args` against any overrides passed to `respond()`.

In [ ]:
swagger_path   = Path("basic_api.swagger.yml")
templates_path = Path("orders.templates.yml")
display(Markdown(f"```yaml\n{templates_path.read_text(encoding='utf-8')}\n```"))

> **Plugin equivalent** — activate the plugin once in `conftest.py`, then reference the contract
> and templates on every marker. See [`example/`](../example/) for full working demos.
>
> ```python
> # conftest.py
> pytest_plugins = ["mockture.pytest_plugin"]
> ```
>
> ```python
> @pytest.mark.mockture(
>     contract="configs/basic_api.openapi.yml",
>     templates="configs/orders.templates.yml",
> )
> def test_something(mockture): ...
> ```

## 3. Template Defaults

We call `respond()` with only the template name, using all default `args` values from the template.
The server is started after `respond()` and stopped immediately after the request is made.

In [ ]:
mock = Mockture(contract_path=str(swagger_path), templates_path=str(templates_path))
mock.respond("create_order_success")   # order_id="ord-default", status="created"
mock.start()

r = httpx.post(mock.url_for("/orders"), json={"item_id": "SKU-1", "quantity": 1}, timeout=5.0)
print("status:", r.status_code)
print("body:  ", r.json())

mock.stop()

> **Plugin equivalent**
>
> ```python
> @pytest.mark.mockture(contract=..., templates=...)
> def test_defaults(mockture):
>     mockture.respond("create_order_success")   # same call, same defaults
>     r = httpx.post(mockture.url_for("/orders"), json={"item_id": "SKU-1", "quantity": 1})
>     assert r.json() == {"order_id": "ord-default", "status": "created"}
> ```

## 4. Arg Overrides and Method Chaining

`respond()` returns `self`, so calls can be chained. Each chained call registers one interaction;
requests are served in registration order (first-in, first-out). We register a 201 response
followed by a 409 response and confirm that two sequential requests receive them in that order.
`assert_called()` raises `AssertionError` if the observed call count does not match the expected value.

In [ ]:
mock = Mockture(contract_path=str(swagger_path), templates_path=str(templates_path))
(
    mock
    .respond("create_order_success", order_id="ord-123", status="queued")
    .respond("create_order_conflict")                                        # 409 defaults
)
mock.start()

r1 = httpx.post(mock.url_for("/orders"), json={"item_id": "A", "quantity": 1}, timeout=5.0)
r2 = httpx.post(mock.url_for("/orders"), json={"item_id": "B", "quantity": 1}, timeout=5.0)
print("call 1:", r1.status_code, r1.json())
print("call 2:", r2.status_code, r2.json())
mock.assert_called("/orders", "POST", 2)

mock.stop()

> **Plugin equivalent** — use `sequence=` to pre-register an ordered list before the test runs:
>
> ```python
> @pytest.mark.mockture(
>     contract=..., templates=...,
>     sequence=["create_order_success", "create_order_conflict"],
> )
> def test_sequence(mockture):
>     r1 = httpx.post(mockture.url_for("/orders"), json={"item_id": "A", "quantity": 1})
>     r2 = httpx.post(mockture.url_for("/orders"), json={"item_id": "B", "quantity": 1})
>     assert r1.status_code == 201
>     assert r2.status_code == 409
>     mockture.assert_called("/orders", "POST", 2)
> ```

## 5. Scoped Context

We use `for_context()` to inject a shared argument across multiple `respond()` calls without
repeating it on each call. Arguments passed explicitly to `ctx.respond()` take priority over
context arguments, which in turn take priority over template defaults.

In [ ]:
mock = Mockture(contract_path=str(swagger_path), templates_path=str(templates_path))

with mock.for_context(order_id="ctx-001") as ctx:
    ctx.respond("create_order_success", status="queued")   # order_id injected from context

mock.start()

r = httpx.post(mock.url_for("/orders"), json={"item_id": "SKU-1", "quantity": 1}, timeout=5.0)
print("body:", r.json())   # order_id from context, status from respond()

mock.stop()

> **Plugin equivalent** — `for_context()` is the same API inside a test body:
>
> ```python
> @pytest.mark.mockture(contract=..., templates=...)
> def test_context(mockture):
>     with mockture.for_context(order_id="ctx-001") as ctx:
>         ctx.respond("create_order_success", status="queued")
>     r = httpx.post(mockture.url_for("/orders"), json={"item_id": "SKU-1", "quantity": 1})
>     assert r.json()["order_id"] == "ctx-001"
> ```

## 6. Inline Scenario Dict

We pass a `dict` directly to `respond()` to register multiple templates in one call. Each key
is a template name; its value is a dict of arg overrides, or `None` to use all defaults.

In [ ]:
mock = Mockture(contract_path=str(swagger_path), templates_path=str(templates_path))
mock.respond({
    "create_order_success": {"order_id": "ord-scen", "status": "processing"},
    "create_order_conflict": None,   # defaults only
})
mock.start()

r1 = httpx.post(mock.url_for("/orders"), json={"item_id": "A", "quantity": 1}, timeout=5.0)
r2 = httpx.post(mock.url_for("/orders"), json={"item_id": "B", "quantity": 1}, timeout=5.0)
print("scenario call 1:", r1.status_code, r1.json())
print("scenario call 2:", r2.status_code, r2.json())

mock.stop()

> **Plugin equivalent** — use `scenario=` to pre-register the dict before the test runs:
>
> ```python
> @pytest.mark.mockture(
>     contract=..., templates=...,
>     scenario={
>         "create_order_success": {"order_id": "ord-scen", "status": "processing"},
>         "create_order_conflict": None,
>     },
> )
> def test_scenario(mockture):
>     r1 = httpx.post(mockture.url_for("/orders"), json={"item_id": "A", "quantity": 1})
>     r2 = httpx.post(mockture.url_for("/orders"), json={"item_id": "B", "quantity": 1})
>     assert r1.status_code == 201
>     assert r2.status_code == 409
> ```

## 7. Contract Validation

Mockture validates interactions against the OpenAPI contract at two distinct phases.

### 7.1 Config-Time Validation

We attempt to configure a template whose response body contains an undeclared field. `respond()`
raises `ContractConfigError` immediately — before `start()` is called — because the body does not
satisfy the schema declared in the contract.

In [ ]:
# Config-time: respond() raises ContractConfigError before start()
mock = Mockture(contract_path=str(swagger_path), templates_path=str(templates_path))
try:
    mock.respond("invalid_response_shape")   # body {"bad_field": "oops"} fails schema
except ContractConfigError as exc:
    print("ContractConfigError (config-time):", exc)

### 7.2 Runtime Strict Mode

With `strict=True` (the default), a request that violates the contract schema is rejected at
runtime with HTTP 500 and is not counted in the call log. We confirm that only the valid request
increments the call count.

In [ ]:
# Runtime strict mode: invalid request -> HTTP 500
mock = Mockture(
    contract_path=str(swagger_path),
    templates_path=str(templates_path),
    strict=True,
)
mock.respond("create_order_success", order_id="ord-strict", status="queued")
mock.start()

valid   = httpx.post(mock.url_for("/orders"), json={"item_id": "SKU-1", "quantity": 1}, timeout=5.0)
invalid = httpx.post(mock.url_for("/orders"), json={"item_id": "",       "quantity": 0}, timeout=5.0)
print("valid:  ", valid.status_code,   valid.json())
print("invalid:", invalid.status_code, invalid.json())

mock.assert_called("/orders", "POST", 1)   # blocked request is not counted
mock.stop()

> **Plugin equivalent** — config-time validation is automatic on `respond()`. Runtime mode is
> controlled by the `strict=` kwarg, which defaults to `True`:
>
> ```python
> @pytest.mark.mockture(contract=..., templates=..., strict=True)
> def test_strict(mockture):
>     mockture.respond("create_order_success", order_id="ord-strict", status="queued")
>     valid   = httpx.post(mockture.url_for("/orders"), json={"item_id": "SKU-1", "quantity": 1})
>     invalid = httpx.post(mockture.url_for("/orders"), json={"item_id": "",       "quantity": 0})
>     assert valid.status_code == 201
>     assert invalid.status_code == 500
>     mockture.assert_called("/orders", "POST", 1)
> ```

## 8. Non-Strict Mode

With `strict=False`, a request that violates the contract schema is recorded as a violation but
the configured response is still returned. We call `assert_no_contract_violations()` after the
request to surface the recorded violation as an `AssertionError`.

In [ ]:
mock = Mockture(
    contract_path=str(swagger_path),
    templates_path=str(templates_path),
    strict=False,
)
mock.respond("create_order_success", order_id="ord-lenient", status="accepted")
mock.start()

invalid = httpx.post(mock.url_for("/orders"), json={"item_id": "", "quantity": 0}, timeout=5.0)
print("non-strict invalid:", invalid.status_code, invalid.json())   # still 201

try:
    mock.assert_no_contract_violations()
except AssertionError as exc:
    print("violations:", exc)

mock.stop()

> **Plugin equivalent**
>
> ```python
> @pytest.mark.mockture(contract=..., templates=..., strict=False)
> def test_non_strict(mockture):
>     mockture.respond("create_order_success", order_id="ord-lenient", status="accepted")
>     invalid = httpx.post(mockture.url_for("/orders"), json={"item_id": "", "quantity": 0})
>     assert invalid.status_code == 201            # response still returned
>     mockture.assert_no_contract_violations()     # raises if any violations were recorded
> ```

## 9. Inspecting Recorded Calls

We use `calls_for(path, method)` to retrieve a `CallView`, then inspect the `CallRecord` fields
of the first recorded call.

- **`CallView`**: a filtered view of all calls for a given path and method. Exposes `.count` and
  `.records`.
- **`CallRecord`**: an immutable record of one request. Fields: `method`, `path`, `json_body`,
  `headers`, `status_code`.

In [ ]:
mock = Mockture(contract_path=str(swagger_path), templates_path=str(templates_path))
mock.respond("create_order_success", order_id="ord-rec", status="queued")
mock.start()

httpx.post(mock.url_for("/orders"), json={"item_id": "SKU-1", "quantity": 2}, timeout=5.0)

view = mock.calls_for("/orders", "POST")
print("call count:     ", view.count)

rec = view.records[0]
print("method:         ", rec.method)
print("path:           ", rec.path)
print("request body:   ", rec.json_body)
print("response status:", rec.status_code)

mock.stop()

## 10. Teardown

We signal the background API server to stop and await its shutdown task.

In [ ]:
server.should_exit = True
await server_task
print("Background API stopped")

## 11. Conclusion

This notebook covered the full `Mockture` server object API:

- Configured responses using named templates with default args.
- Overrode template args at `respond()` time and chained multiple interactions.
- Shared args across calls with `for_context()`.
- Registered multiple templates at once with an inline scenario dict.
- Observed config-time `ContractConfigError` and runtime strict-mode HTTP 500 rejection.
- Accumulated violations in non-strict mode and surfaced them with
  `assert_no_contract_violations()`.
- Inspected recorded call metadata via `calls_for()` and `CallRecord`.

Each feature maps directly to the pytest plugin: `respond()` works identically inside test
bodies, and `sequence=`, `scenario=`, and `strict=` on the marker pre-configure the same
operations before the test function runs. See [`example/`](../example/) for end-to-end plugin
usage across multiple API configurations.